# 06 ML 因子分析

**专门面向机器学习的因子筛选与诊断。**  
与 `ml/` 模块深度联动：复用 `ml.features.build_features`、`ml.labels.build_labels`、`ml.train.load_data`、`ml.evaluate` 等组件，
确保此 notebook 分析的因子和特征与实际训练流水线完全一致。

---

**分析流程：**

1. 加载数据与配置（直接读 `ml/config.yaml`）
2. 构建 ML 特征矩阵（含滞后、滚动统计）
3. 基础因子 IC 筛选（对齐 ML 标签窗口）
4. 特征相关性分析与冗余检测
5. LightGBM 特征重要性分析
6. SHAP 值解读
7. 特征稳定性（Walk-Forward IC）
8. 自动化特征筛选流水线
9. 筛选前后模型对比

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path

from ml.train import load_config, load_data, _ts
from ml.features import build_features, _default_factors
from ml.labels import build_labels
from ml.evaluate import rank_ic, oof_metrics, sharpe_from_pred, freq_to_annual_factor

CFG_PATH = Path('..') / 'ml' / 'config.yaml'
cfg = load_config(CFG_PATH)

freq = cfg['data'].get('freq', '1D')
bars_per_year = freq_to_annual_factor(freq)
forward_bars = cfg['labels'].get('forward_bars', cfg['labels'].get('forward_days', 1))

print('ML 配置摘要:')
print(f'  品种: {cfg["data"]["product"]}')
print(f'  时间: {cfg["data"]["start_date"]} ~ {cfg["data"]["end_date"]}')
print(f'  频率: {freq}  (年化系数={bars_per_year})')
print(f'  标签: forward={forward_bars} bars  type={cfg["labels"]["label_type"]}')
print(f'  滞后: {cfg["features"]["lags"]}')
print(f'  滚动: {cfg["features"]["rolling_windows"]}')
print(f'  CV:   {cfg["training"]["n_splits"]} 折  gap={cfg["training"]["gap"]}')

## 1. 加载数据 & 构建 ML 特征

In [ ]:
klines, klines_clean = load_data(cfg)

X = build_features(klines, klines_clean, cfg['features'], freq=freq)
y = build_labels(klines, cfg['labels'], freq=freq)

common = X.dropna(how='all').index.intersection(y.dropna().index)
X, y = X.loc[common], y.loc[common]

print(f'\n特征矩阵: {X.shape}  (bars x 特征)')
print(f'标签样本: {len(y)}')
print(f'有效日期: {X.index[0]} ~ {X.index[-1]}')
print(f'频率: {freq}')
print(f'\n特征列（前20个）: {X.columns[:20].tolist()}')
print(f'缺失率: {X.isna().mean().mean():.2%}')

## 2. 因子 IC 分析（对齐 ML 标签窗口）

使用与 `ml/config.yaml` 相同的 `forward_days` 计算 Rank IC，确保因子筛选标准与模型训练目标一致。

In [ ]:
def calc_feature_ic(
    X: pd.DataFrame,
    y: pd.Series,
    rolling_window: int = 20,
) -> pd.DataFrame:
    """计算每个特征与 ML 标签的 Rank IC / ICIR。"""
    records = {}
    for col in X.columns:
        aligned = pd.concat([X[col], y], axis=1).dropna()
        if len(aligned) < 10:
            continue
        rho, pval = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])

        rolling_ic = []
        for i in range(rolling_window, len(aligned)):
            w = aligned.iloc[i - rolling_window:i]
            r, _ = stats.spearmanr(w.iloc[:, 0], w.iloc[:, 1])
            rolling_ic.append(r)

        if len(rolling_ic) > 1:
            ric = pd.Series(rolling_ic)
            icir = ric.mean() / ric.std() if ric.std() > 0 else np.nan
            ic_pos_pct = (ric > 0).mean()
        else:
            icir = ic_pos_pct = np.nan

        records[col] = {
            'IC': rho, 'p_value': pval, 'ICIR': icir,
            'IC_pos_pct': ic_pos_pct, '|IC|': abs(rho),
        }

    return pd.DataFrame(records).T.sort_values('|IC|', ascending=False)


ic_df = calc_feature_ic(X, y)

print(f'Forward = {forward_bars} bars ({freq}) 的特征 IC 排名 (Top 20):\n')
print(ic_df.head(20).round(4).to_string())

In [ ]:
top30 = ic_df.head(30)
colors = ['#d62728' if v < 0 else '#1f77b4' for v in top30['IC']]

fig = go.Figure(go.Bar(x=top30.index, y=top30['IC'], marker_color=colors))
fig.add_hline(y=0.02, line_dash='dash', line_color='green', annotation_text='IC=0.02')
fig.add_hline(y=-0.02, line_dash='dash', line_color='green')
fig.update_layout(
    title=f'ML 特征 Rank IC Top 30（forward {forward_bars} bars @ {freq}）',
    xaxis_title='特征', yaxis_title='IC', height=420,
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
def classify_feature(name: str) -> str:
    if '_lag' in name:
        return 'Lag'
    if '_rmean' in name:
        return 'RollMean'
    if '_rstd' in name:
        return 'RollStd'
    return 'Base'

ic_df['type'] = ic_df.index.map(classify_feature)

type_stats = ic_df.groupby('type').agg(
    count=('IC', 'size'),
    mean_absIC=('|IC|', 'mean'),
    max_absIC=('|IC|', 'max'),
    mean_ICIR=('ICIR', 'mean'),
).round(4)

print('各特征类型 IC 统计:')
print(type_stats.to_string())

## 3. 特征相关性 & 冗余检测

高相关特征组（|r| > 0.7）内只保留 |IC| 最高的代表，减少冗余、降低过拟合风险。

In [ ]:
base_factors = [f.name for f in _default_factors()]
base_cols = [c for c in X.columns if c in base_factors]

corr = X[base_cols].dropna().corr(method='spearman').round(2)

fig = px.imshow(
    corr, text_auto=True, color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, aspect='auto',
    title='基础因子 Spearman 相关性矩阵',
)
fig.update_layout(height=600, width=700)
fig.show()

In [ ]:
def find_redundant_pairs(
    X: pd.DataFrame,
    ic_df: pd.DataFrame,
    corr_threshold: float = 0.7,
) -> pd.DataFrame:
    """找出 |相关系数| > 阈值的特征对，标记可去除的低 IC 一方。"""
    corr_full = X.corr(method='spearman')
    pairs = []
    for i, c1 in enumerate(corr_full.columns):
        for j, c2 in enumerate(corr_full.columns):
            if i >= j:
                continue
            r = corr_full.iloc[i, j]
            if pd.notna(r) and abs(r) >= corr_threshold:
                ic1 = ic_df.loc[c1, '|IC|'] if c1 in ic_df.index else 0
                ic2 = ic_df.loc[c2, '|IC|'] if c2 in ic_df.index else 0
                drop = c2 if ic1 >= ic2 else c1
                pairs.append({
                    'feat_1': c1, 'feat_2': c2,
                    'corr': round(r, 3),
                    'IC_1': round(ic1, 4), 'IC_2': round(ic2, 4),
                    'drop': drop,
                })
    if not pairs:
        return pd.DataFrame(columns=['feat_1', 'feat_2', 'corr', 'IC_1', 'IC_2', 'drop'])
    return pd.DataFrame(pairs).sort_values('corr', key=abs, ascending=False)


redundant = find_redundant_pairs(X, ic_df, corr_threshold=0.7)
print(f'|相关系数| > 0.7 的特征对: {len(redundant)} 个')
if len(redundant) > 0:
    print(redundant.head(20).to_string(index=False))
else:
    print('（无高度冗余特征对）')

to_drop_corr = set(redundant['drop'].tolist()) if len(redundant) > 0 else set()
print(f'\n建议去除（冗余）: {len(to_drop_corr)} 个特征')

## 4. LightGBM 特征重要性

使用完整 ML 流水线训练，提取 Gain 和 Split 两种重要性度量。

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

tcfg = cfg['training']
mcfg = cfg['model']
params = mcfg['params']

tscv = TimeSeriesSplit(
    n_splits=tcfg['n_splits'],
    gap=tcfg.get('gap', 1),
    test_size=tcfg.get('test_size', None),
    max_train_size=tcfg.get('max_train_size', None),
)

gain_list, split_list = [], []
oof_preds = pd.Series(np.nan, index=y.index, name='oof_pred')
fold_indices = []

print(f'训练 {tcfg["n_splits"]} 折模型（freq={freq}）...')
for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=False)

    model = lgb.train(
        params, dtrain,
        num_boost_round=mcfg.get('num_boost_round', 500),
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(mcfg.get('early_stopping_rounds', 50), verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    preds = model.predict(X_val)
    oof_preds.iloc[val_idx] = preds
    fold_indices.append((train_idx, val_idx))

    gain_list.append(pd.Series(model.feature_importance('gain'), index=X.columns))
    split_list.append(pd.Series(model.feature_importance('split'), index=X.columns))

    fold_ic = rank_ic(y_val, pd.Series(preds, index=y_val.index))
    print(f'  Fold {fold}  train={len(train_idx)}  val={len(val_idx)}  '
          f'IC={fold_ic:.4f}  trees={model.best_iteration}')

gain_mean = pd.concat(gain_list, axis=1).mean(axis=1).sort_values(ascending=False)
split_mean = pd.concat(split_list, axis=1).mean(axis=1).sort_values(ascending=False)

metrics = oof_metrics(y, oof_preds, fold_indices, bars_per_year=bars_per_year)
print(f'\nOOF 全量模型: IC={metrics["IC"]}  ICIR={metrics["ICIR"]}  '
      f'Sharpe={metrics["Sharpe"]}  MaxDD={metrics["MaxDrawdown"]}')

In [ ]:
top_n = min(30, len(gain_mean))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Gain 重要性', 'Split 重要性'])

top_gain = gain_mean.head(top_n)
fig.add_trace(
    go.Bar(x=top_gain.index, y=top_gain.values, marker_color='steelblue',
           showlegend=False),
    row=1, col=1,
)

top_split = split_mean.head(top_n)
fig.add_trace(
    go.Bar(x=top_split.index, y=top_split.values, marker_color='coral',
           showlegend=False),
    row=1, col=2,
)

fig.update_layout(
    title=f'LightGBM 特征重要性 Top {top_n}（{tcfg["n_splits"]} 折平均）',
    height=450,
)
fig.update_xaxes(tickangle=-45)
fig.show()

In [ ]:
gain_all = pd.concat(gain_list, axis=1)
gain_cv = (gain_all.std(axis=1) / gain_all.mean(axis=1)).rename('CV')
gain_summary = pd.DataFrame({
    'mean_gain': gain_mean,
    'CV': gain_cv,
}).sort_values('mean_gain', ascending=False)

print('特征重要性稳定性（Top 20，CV 越低越稳定）:')
print(gain_summary.head(20).round(3).to_string())

## 5. SHAP 值分析

SHAP 揭示每个特征对预测值的边际贡献方向和大小，帮助理解模型在怎样使用因子。

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('shap 未安装，跳过 SHAP 分析。安装: pip install shap')

if HAS_SHAP:
    last_val_idx = fold_indices[-1][1]
    X_shap = X.iloc[last_val_idx]

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    mean_abs_shap = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=X.columns,
    ).sort_values(ascending=False)

    print('SHAP 特征排名 Top 20:')
    print(mean_abs_shap.head(20).round(6).to_string())

In [ ]:
if HAS_SHAP:
    top20_names = mean_abs_shap.index[:20].tolist()
    fig = go.Figure()
    for yi, feat in enumerate(top20_names[::-1]):
        feat_idx = list(X.columns).index(feat)
        sv = shap_values[:, feat_idx]
        fv = X_shap[feat].values.astype(float)
        fv_filled = np.where(np.isnan(fv), np.nanmedian(fv), fv)
        lo, hi = np.nanmin(fv_filled), np.nanmax(fv_filled)
        fv_norm = (fv_filled - lo) / (hi - lo + 1e-10)
        colors = [f'rgb({int(255*v)},0,{int(255*(1-v))})' for v in fv_norm]
        jitter = np.random.default_rng(42).uniform(-0.25, 0.25, size=len(sv))
        fig.add_trace(go.Scatter(
            x=sv, y=yi + jitter,
            mode='markers',
            marker=dict(size=3, color=colors, opacity=0.5),
            showlegend=False, hovertext=[f'{feat}={v:.4f}' for v in fv],
        ))
    fig.update_layout(
        title='SHAP Beeswarm（Top 20 特征，红=高值 蓝=低值）',
        yaxis=dict(tickvals=list(range(len(top20_names))),
                   ticktext=top20_names[::-1]),
        xaxis_title='SHAP value', height=600,
    )
    fig.show()

    top20_shap = mean_abs_shap.head(20)
    fig2 = go.Figure(go.Bar(
        x=top20_shap.index, y=top20_shap.values,
        marker_color='mediumpurple',
    ))
    fig2.update_layout(
        title='Mean |SHAP| Top 20', height=400,
        xaxis_tickangle=-45, yaxis_title='Mean |SHAP|',
    )
    fig2.show()

In [ ]:
if HAS_SHAP:
    top4 = mean_abs_shap.index[:4].tolist()
    fig = make_subplots(rows=2, cols=2, subplot_titles=top4)
    for i, feat in enumerate(top4):
        r, c = divmod(i, 2)
        feat_idx = list(X.columns).index(feat)
        fig.add_trace(
            go.Scatter(
                x=X_shap[feat].values,
                y=shap_values[:, feat_idx],
                mode='markers', marker=dict(size=3, opacity=0.4),
                showlegend=False,
            ),
            row=r + 1, col=c + 1,
        )
        fig.update_xaxes(title_text=feat, row=r + 1, col=c + 1)
        fig.update_yaxes(title_text='SHAP', row=r + 1, col=c + 1)
    fig.update_layout(title='SHAP Dependence（Top 4 特征）', height=600)
    fig.show()

## 6. 特征稳定性：Walk-Forward IC

将时间轴切分为多个窗口，计算每个窗口内各特征的 IC。  
稳定因子 = 各窗口 IC 符号一致、均值显著。

In [ ]:
def walk_forward_ic(
    X: pd.DataFrame,
    y: pd.Series,
    n_windows: int = 6,
) -> pd.DataFrame:
    """在 n_windows 个等长时间段内分别计算特征 IC。"""
    idx = X.dropna(how='all').index.intersection(y.dropna().index)
    splits = np.array_split(idx, n_windows)
    records = {}
    for wi, split_idx in enumerate(splits):
        x_w = X.loc[split_idx]
        y_w = y.loc[split_idx]
        label = f'{str(split_idx[0].date())[:7]}~{str(split_idx[-1].date())[:7]}'
        for col in X.columns:
            aligned = pd.concat([x_w[col], y_w], axis=1).dropna()
            if len(aligned) < 10:
                continue
            rho, _ = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
            records.setdefault(col, {})[label] = rho
    return pd.DataFrame(records).T


wf_ic = walk_forward_ic(X, y, n_windows=6)

wf_ic['mean_IC'] = wf_ic.mean(axis=1)
wf_ic['IC_std'] = wf_ic.iloc[:, :-1].std(axis=1)
wf_ic['sign_consistency'] = wf_ic.iloc[:, :-2].apply(
    lambda row: (row > 0).sum() / len(row) if (row > 0).sum() >= len(row) / 2
    else (row < 0).sum() / len(row), axis=1,
)
wf_ic = wf_ic.sort_values('sign_consistency', ascending=False)

print('Walk-Forward IC 稳定性 Top 20:')
print(wf_ic[['mean_IC', 'IC_std', 'sign_consistency']].head(20).round(4).to_string())

In [ ]:
top15_stable = wf_ic.head(15).index.tolist()
window_cols = [c for c in wf_ic.columns if '~' in c]

fig = px.imshow(
    wf_ic.loc[top15_stable, window_cols].round(3).astype(float),
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-0.15, zmax=0.15,
    aspect='auto',
    title='Walk-Forward IC 热图（Top 15 稳定特征）',
)
fig.update_layout(height=500, width=800)
fig.show()

## 7. 综合排名：IC × 重要性 × 稳定性

将三维指标标准化后加权求和，给出因子的综合 ML 评分。

In [ ]:
def composite_score(
    ic_df: pd.DataFrame,
    gain_mean: pd.Series,
    wf_ic: pd.DataFrame,
    w_ic: float = 0.3,
    w_gain: float = 0.4,
    w_stability: float = 0.3,
) -> pd.DataFrame:
    """综合评分 = w_ic * |IC|_rank + w_gain * gain_rank + w_stability * stability_rank。"""
    common = ic_df.index.intersection(gain_mean.index).intersection(wf_ic.index)
    df = pd.DataFrame(index=common)
    df['|IC|'] = ic_df.loc[common, '|IC|']
    df['Gain'] = gain_mean.loc[common]
    df['Stability'] = wf_ic.loc[common, 'sign_consistency']

    for col in ['|IC|', 'Gain', 'Stability']:
        df[f'{col}_pct'] = df[col].rank(pct=True)

    df['score'] = (
        w_ic * df['|IC|_pct'] +
        w_gain * df['Gain_pct'] +
        w_stability * df['Stability_pct']
    )
    return df.sort_values('score', ascending=False)


scores = composite_score(ic_df, gain_mean, wf_ic)

print('综合评分 Top 30（权重: IC=0.3, Gain=0.4, Stability=0.3）:')
print(scores[['|IC|', 'Gain', 'Stability', 'score']].head(30).round(4).to_string())

In [ ]:
plot_df = scores.head(30).reset_index().rename(columns={'index': 'feature'})

fig = px.scatter(
    plot_df, x='|IC|', y='Gain',
    size='Stability', size_max=25,
    color='score', color_continuous_scale='Viridis',
    text='feature',
    title='Top 30 特征：|IC| vs Gain（气泡=稳定性，颜色=综合评分）',
)
fig.update_traces(textposition='top center', textfont_size=8)
fig.update_layout(height=550)
fig.show()

## 8. 自动化特征筛选流水线

三步过滤：
1. **IC 过滤**：去除 p > 0.2 且 |IC| < 0.01 的纯噪声
2. **相关性过滤**：高相关组内只保留 IC 最高的
3. **重要性过滤**：去除 Gain 重要性低于中位数的特征

In [ ]:
def feature_selection_pipeline(
    X: pd.DataFrame,
    ic_df: pd.DataFrame,
    gain_mean: pd.Series,
    ic_threshold: float = 0.01,
    p_threshold: float = 0.2,
    corr_threshold: float = 0.7,
    importance_quantile: float = 0.5,
) -> dict:
    """三步特征筛选，返回每步的保留/去除特征。"""
    all_features = set(X.columns)
    result = {'step0_total': len(all_features)}

    # Step 1: IC + p-value
    noise = ic_df[
        (ic_df['|IC|'] < ic_threshold) & (ic_df['p_value'] > p_threshold)
    ].index.tolist()
    keep_1 = all_features - set(noise)
    result['step1_ic_filter'] = {
        'removed': len(noise), 'kept': len(keep_1),
        'removed_features': sorted(noise),
    }

    # Step 2: corr（逐对计算，不整体 dropna）
    X_sub = X[[c for c in X.columns if c in keep_1]]
    corr_full = X_sub.corr(method='spearman')
    to_drop = set()
    for i, c1 in enumerate(corr_full.columns):
        if c1 in to_drop:
            continue
        for j, c2 in enumerate(corr_full.columns):
            if i >= j or c2 in to_drop:
                continue
            r = corr_full.iloc[i, j]
            if pd.notna(r) and abs(r) >= corr_threshold:
                ic1 = ic_df.loc[c1, '|IC|'] if c1 in ic_df.index else 0
                ic2 = ic_df.loc[c2, '|IC|'] if c2 in ic_df.index else 0
                to_drop.add(c2 if ic1 >= ic2 else c1)
    keep_2 = keep_1 - to_drop
    result['step2_corr_filter'] = {
        'removed': len(to_drop), 'kept': len(keep_2),
        'removed_features': sorted(to_drop),
    }

    # Step 3: importance
    gain_sub = gain_mean.loc[gain_mean.index.isin(keep_2)]
    threshold = gain_sub.quantile(importance_quantile)
    low_imp = gain_sub[gain_sub < threshold].index.tolist()
    keep_3 = keep_2 - set(low_imp)
    result['step3_importance_filter'] = {
        'removed': len(low_imp), 'kept': len(keep_3),
        'removed_features': sorted(low_imp),
    }

    result['selected_features'] = sorted(keep_3)
    return result


selection = feature_selection_pipeline(X, ic_df, gain_mean)

print(f'特征筛选流水线:')
print(f'  Step 0: 总特征         {selection["step0_total"]}')
print(f'  Step 1: IC 过滤后      {selection["step1_ic_filter"]["kept"]}  '
      f'(-{selection["step1_ic_filter"]["removed"]})')
print(f'  Step 2: 相关性过滤后   {selection["step2_corr_filter"]["kept"]}  '
      f'(-{selection["step2_corr_filter"]["removed"]})')
print(f'  Step 3: 重要性过滤后   {selection["step3_importance_filter"]["kept"]}  '
      f'(-{selection["step3_importance_filter"]["removed"]})')
print(f'\n最终保留 {len(selection["selected_features"])} 个特征')

In [ ]:
steps = ['全部特征', 'IC 过滤', '相关性过滤', '重要性过滤']
counts = [
    selection['step0_total'],
    selection['step1_ic_filter']['kept'],
    selection['step2_corr_filter']['kept'],
    selection['step3_importance_filter']['kept'],
]

fig = go.Figure(go.Funnel(
    y=steps, x=counts,
    textinfo='value+percent initial',
    marker=dict(color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA']),
))
fig.update_layout(title='特征筛选漏斗', height=350)
fig.show()

## 9. 筛选前后模型对比

使用相同的训练配置，对比全特征 vs 筛选后特征的 OOF 表现。

In [ ]:
selected = selection['selected_features']
X_sel = X[selected]

oof_sel = pd.Series(np.nan, index=y.index, name='oof_sel')
fold_indices_sel = []

print(f'使用 {len(selected)} 个筛选特征重新训练...')
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sel), start=1):
    X_tr, X_val = X_sel.iloc[train_idx], X_sel.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=False)

    model_sel = lgb.train(
        params, dtrain,
        num_boost_round=mcfg.get('num_boost_round', 500),
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(mcfg.get('early_stopping_rounds', 50), verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    preds = model_sel.predict(X_val)
    oof_sel.iloc[val_idx] = preds
    fold_indices_sel.append((train_idx, val_idx))

    fold_ic = rank_ic(y_val, pd.Series(preds, index=y_val.index))
    print(f'  Fold {fold}  IC={fold_ic:.4f}  trees={model_sel.best_iteration}')

metrics_sel = oof_metrics(y, oof_sel, fold_indices_sel, bars_per_year=bars_per_year)

In [ ]:
compare = pd.DataFrame({
    '全特征': {
        '特征数': X.shape[1],
        'IC': metrics['IC'],
        'ICIR': metrics['ICIR'],
        'Sharpe': metrics['Sharpe'],
        'MaxDD': metrics['MaxDrawdown'],
    },
    '筛选后': {
        '特征数': len(selected),
        'IC': metrics_sel['IC'],
        'ICIR': metrics_sel['ICIR'],
        'Sharpe': metrics_sel['Sharpe'],
        'MaxDD': metrics_sel['MaxDrawdown'],
    },
})
compare['差异'] = compare['筛选后'] - compare['全特征']

print('全特征 vs 筛选后 模型对比:')
print(compare.to_string())

In [ ]:
def build_nav(y_true, y_pred, thr=0.2):
    df = pd.concat([y_true, y_pred], axis=1).dropna()
    df.columns = ['ret', 'pred']
    df['signal'] = 0
    df.loc[df['pred'] >= df['pred'].quantile(1 - thr), 'signal'] = 1
    df.loc[df['pred'] <= df['pred'].quantile(thr), 'signal'] = -1
    df['ls_ret'] = df['signal'].shift(1) * df['ret']
    return (1 + df['ls_ret'].fillna(0)).cumprod()

nav_full = build_nav(y, oof_preds)
nav_sel = build_nav(y, oof_sel)
nav_bh = (1 + y.fillna(0)).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=nav_full.index, y=nav_full.values,
                         name=f'全特征 ({X.shape[1]})', line=dict(width=2)))
fig.add_trace(go.Scatter(x=nav_sel.index, y=nav_sel.values,
                         name=f'筛选后 ({len(selected)})', line=dict(width=2)))
fig.add_trace(go.Scatter(x=nav_bh.index, y=nav_bh.values,
                         name='买入持有', line=dict(dash='dot', color='gray')))
fig.update_layout(
    title='OOF 多空净值对比：全特征 vs 筛选后 vs 买入持有',
    height=420, yaxis_title='净值', xaxis_title='日期',
)
fig.show()

In [ ]:
fold_labels = [f'Fold {i+1}' for i in range(len(metrics['fold_ICs']))]

fig = go.Figure()
fig.add_trace(go.Bar(
    name='全特征', x=fold_labels, y=metrics['fold_ICs'],
    marker_color='steelblue', opacity=0.7,
))
fig.add_trace(go.Bar(
    name='筛选后', x=fold_labels, y=metrics_sel['fold_ICs'],
    marker_color='coral', opacity=0.7,
))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(
    title='逐折 IC 对比', barmode='group', height=350,
    yaxis_title='Rank IC',
)
fig.show()

## 10. 导出筛选结果供 ML 流水线使用

将筛选后的特征列表保存为 YAML，可直接在 `ml/config.yaml` 中引用或在 `build_features` 中传入。

In [ ]:
import yaml

export = {
    'selected_features': selection['selected_features'],
    'n_features': len(selection['selected_features']),
    'pipeline_config': {
        'ic_threshold': 0.01,
        'p_threshold': 0.2,
        'corr_threshold': 0.7,
        'importance_quantile': 0.5,
    },
    'metrics_comparison': {
        'full': {
            'n_features': int(X.shape[1]),
            'IC': metrics['IC'],
            'ICIR': metrics['ICIR'],
            'Sharpe': metrics['Sharpe'],
        },
        'selected': {
            'n_features': len(selected),
            'IC': metrics_sel['IC'],
            'ICIR': metrics_sel['ICIR'],
            'Sharpe': metrics_sel['Sharpe'],
        },
    },
}

output_path = Path('..') / 'ml' / 'selected_features.yaml'
with open(output_path, 'w', encoding='utf-8') as f:
    yaml.dump(export, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f'已导出至 {output_path}')
print(f'\n使用方式:')
print(f'  # 在训练脚本中加载筛选特征')
print(f'  import yaml')
print(f'  with open("ml/selected_features.yaml") as f:')
print(f'      sel = yaml.safe_load(f)')
print(f'  X_selected = X[sel["selected_features"]]')

## 11. 标签 A/B 对比：Regression vs Triple Barrier

使用相同的特征集（筛选后），分别用 regression 和 triple_barrier 标签训练 LightGBM，对比 OOF 表现。
Triple Barrier 是分类任务（-1/0/1），使用 `multiclass` 目标函数，预测概率后转为信号。

In [ ]:
# ── A 组：Regression 标签（已有结果复用） ──────────────────────────
# 使用筛选后特征的 oof_sel / metrics_sel 作为 regression 基线

# ── B 组：Triple Barrier 标签 ───────────────────────────────────
tb_cfg = {
    'label_type': 'triple_barrier',
    'forward_bars': cfg['labels'].get('forward_bars', cfg['labels'].get('forward_days', 20)),
    'atr_period': cfg['labels'].get('atr_period', 14),
    'atr_multiplier': cfg['labels'].get('atr_multiplier', 1.5),
}
y_tb = build_labels(klines, tb_cfg, freq=freq)

# 对齐特征与 TB 标签
common_tb = X_sel.dropna(how='all').index.intersection(y_tb.dropna().index)
X_tb, y_tb_aligned = X_sel.loc[common_tb], y_tb.loc[common_tb]

print(f'Triple Barrier 标签分布:')
print(y_tb_aligned.value_counts().sort_index().to_string())
print(f'\n有效样本: {len(y_tb_aligned)}')
print(f'参数: forward_bars={tb_cfg["forward_bars"]}  '
      f'ATR_period={tb_cfg["atr_period"]}  '
      f'ATR_mult={tb_cfg["atr_multiplier"]}')

In [ ]:
# 训练 Triple Barrier 分类模型
# 标签 -1/0/1 映射为 0/1/2 供 LightGBM multiclass 使用
y_tb_cls = y_tb_aligned.map({-1.0: 0, 0.0: 1, 1.0: 2}).astype(int)
n_classes = y_tb_cls.nunique()

params_tb = {
    'objective': 'multiclass',
    'num_class': n_classes,
    'metric': 'multi_logloss',
    'num_leaves': params.get('num_leaves', 31),
    'learning_rate': params.get('learning_rate', 0.05),
    'feature_fraction': params.get('feature_fraction', 0.8),
    'bagging_fraction': params.get('bagging_fraction', 0.8),
    'bagging_freq': params.get('bagging_freq', 5),
    'min_child_samples': params.get('min_child_samples', 20),
    'verbose': -1,
}

tscv_tb = TimeSeriesSplit(
    n_splits=tcfg['n_splits'],
    gap=max(tcfg.get('gap', 1), tb_cfg['forward_bars']),
)

oof_tb_proba = np.full((len(y_tb_cls), n_classes), np.nan)
fold_indices_tb = []

print(f'训练 Triple Barrier 分类模型（{tcfg["n_splits"]} 折）...')
for fold, (train_idx, val_idx) in enumerate(tscv_tb.split(X_tb), start=1):
    X_tr, X_val = X_tb.iloc[train_idx], X_tb.iloc[val_idx]
    y_tr, y_val = y_tb_cls.iloc[train_idx], y_tb_cls.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain, free_raw_data=False)

    model_tb = lgb.train(
        params_tb, dtrain,
        num_boost_round=mcfg.get('num_boost_round', 500),
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(mcfg.get('early_stopping_rounds', 50), verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    proba = model_tb.predict(X_val)  # shape (n_val, n_classes)
    oof_tb_proba[val_idx] = proba
    fold_indices_tb.append((train_idx, val_idx))

    pred_cls = np.argmax(proba, axis=1)
    acc = (pred_cls == y_val.values).mean()
    print(f'  Fold {fold}  train={len(train_idx)}  val={len(val_idx)}  '
          f'acc={acc:.3f}  trees={model_tb.best_iteration}')

# 将概率转为连续信号：P(触止盈) - P(触止损)
# class 0=-1(止损), class 1=0(超时), class 2=+1(止盈)
oof_tb_signal = pd.Series(
    oof_tb_proba[:, 2] - oof_tb_proba[:, 0],
    index=y_tb_cls.index, name='tb_signal',
)
oof_tb_pred_cls = pd.Series(
    np.argmax(oof_tb_proba, axis=1) - 1,  # 映射回 -1/0/1
    index=y_tb_cls.index, name='tb_pred',
)

# 分类指标
from sklearn.metrics import classification_report, accuracy_score
valid_mask = ~np.isnan(oof_tb_proba[:, 0])
y_true_valid = y_tb_cls[valid_mask]
y_pred_valid = oof_tb_pred_cls[valid_mask].astype(int) + 1  # 回到 0/1/2

print(f'\nOOF 分类报告:')
print(classification_report(
    y_true_valid, y_pred_valid,
    target_names=['止损(-1)', '超时(0)', '止盈(+1)'],
    digits=3,
))

In [ ]:
# ── A/B 对比：用 TB 信号构造多空净值，与 Regression 对比 ──────────

# TB 净值：用 P(止盈)-P(止损) 作为连续信号，取 top/bottom 20% 做多空
def build_nav_from_signal(signal, ret, thr=0.2):
    """通用净值构造：signal 为连续信号，ret 为对应的实际收益。"""
    df = pd.concat([ret.rename('ret'), signal.rename('sig')], axis=1).dropna()
    df['pos'] = 0
    df.loc[df['sig'] >= df['sig'].quantile(1 - thr), 'pos'] = 1
    df.loc[df['sig'] <= df['sig'].quantile(thr), 'pos'] = -1
    df['ls_ret'] = df['pos'].shift(1) * df['ret']
    return (1 + df['ls_ret'].fillna(0)).cumprod()

# Regression 组的实际收益
y_reg_common = y.loc[y.index.isin(oof_sel.dropna().index)]
nav_reg = build_nav_from_signal(oof_sel.dropna(), y_reg_common)

# TB 组的实际收益：用 regression 标签的收益率衡量（公平对比）
y_ret_for_tb = y.loc[y.index.isin(oof_tb_signal.dropna().index)]
nav_tb = build_nav_from_signal(oof_tb_signal.dropna(), y_ret_for_tb)

# 买入持有
y_bh = y.loc[y.index.isin(nav_reg.index.union(nav_tb.index))]
nav_bh = (1 + y_bh.fillna(0)).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=nav_reg.index, y=nav_reg.values,
                         name='Regression', line=dict(width=2, color='steelblue')))
fig.add_trace(go.Scatter(x=nav_tb.index, y=nav_tb.values,
                         name='Triple Barrier', line=dict(width=2, color='darkorange')))
fig.add_trace(go.Scatter(x=nav_bh.index, y=nav_bh.values,
                         name='买入持有', line=dict(dash='dot', color='gray')))
fig.update_layout(
    title='A/B 对比：Regression vs Triple Barrier 多空净值',
    height=420, yaxis_title='净值', xaxis_title='日期',
)
fig.show()

In [ ]:
# ── 汇总对比表 ─────────────────────────────────────────────────

def nav_stats(nav, bars_per_year):
    """从净值序列计算年化收益、Sharpe、最大回撤。"""
    ret = nav.pct_change().dropna()
    if len(ret) < 2 or ret.std() == 0:
        return {'年化收益': np.nan, 'Sharpe': np.nan, '最大回撤': np.nan, '胜率': np.nan}
    ann_ret = ret.mean() * bars_per_year
    sharpe = ret.mean() / ret.std() * np.sqrt(bars_per_year)
    mdd = ((nav.cummax() - nav) / nav.cummax()).max()
    win_rate = (ret > 0).mean()
    return {
        '年化收益': f'{ann_ret:.2%}',
        'Sharpe': f'{sharpe:.3f}',
        '最大回撤': f'{mdd:.2%}',
        '胜率': f'{win_rate:.1%}',
    }

ab_compare = pd.DataFrame({
    'Regression': nav_stats(nav_reg, bars_per_year),
    'Triple Barrier': nav_stats(nav_tb, bars_per_year),
})

print('A/B 对比汇总（多空策略，同一特征集，同一时间段）:')
print(ab_compare.to_string())

# Rank IC 对比（TB 信号 vs 实际收益率的 IC）
ic_reg = rank_ic(y_reg_common, oof_sel.dropna())
ic_tb = rank_ic(y_ret_for_tb, oof_tb_signal.dropna())
print(f'\nRank IC:  Regression={ic_reg:.4f}  Triple Barrier={ic_tb:.4f}')

---

## Summary

| 分析维度 | 核心输出 | 用途 |
|----------|----------|------|
| IC 分析 | 特征预测力排名 | 过滤纯噪声特征 |
| 相关性分析 | 冗余特征对 | 去除高度共线的特征 |
| LightGBM 重要性 | Gain/Split 排名 | 识别模型实际使用的特征 |
| SHAP | 方向 + 边际贡献 | 理解模型如何使用因子 |
| Walk-Forward IC | 时间稳定性 | 筛除阶段性有效的虚假因子 |
| 综合评分 | IC×重要性×稳定性 | 多维度排名 |
| 自动筛选 | 三步漏斗 | 生产可用的特征列表 |
| 模型对比 | 全集 vs 精选 | 验证筛选不损失效果 |